# Zom Nom Asset Foundry — Colab GPU Worker
This notebook consumes `AssetFoundry/asset_manifest.json`, generates selected 3D proposals, refines topology, creates PBR maps/LODs/collisions, and uploads a versioned batch for GitHub validation.

In [ ]:
REPOSITORY = 'ornab74/zom-nom-defense'
BRANCH = 'agent/coherent-zombie-navigation-overhaul'
ASSET_IDS = ['pool_house_shell', 'pool_full', 'muscle_car_survivor', 'zombie_grunt']
QUALITY = 'production'
GCS_BUCKET = ''
JOB_ID = ''  # Set a stable ID such as 2026-07-26-pool-pass-01

In [ ]:
!git clone --depth 1 --branch {BRANCH} https://github.com/{REPOSITORY}.git /content/zom-nom-defense
%cd /content/zom-nom-defense
!python AssetFoundry/tools/manifest_cli.py validate
!pip -q install trimesh pygltflib pymeshlab pillow numpy scipy networkx

## Generator adapter
Install the selected image-to-3D backend in this section. Keep the rest of the notebook backend-neutral: the adapter must return a textured proposal mesh plus provenance metadata. Production candidates include Hunyuan3D, TRELLIS, Stable Fast 3D, or a private Flow-Matching DiT checkpoint.

In [ ]:
from pathlib import Path
import json, time, uuid
ROOT = Path('/content/zom-nom-defense')
MANIFEST = json.loads((ROOT/'AssetFoundry/asset_manifest.json').read_text())
SPECS = {a['id']: a for a in MANIFEST['assets']}
JOB_ID = JOB_ID or f"colab-{int(time.time())}-{uuid.uuid4().hex[:6]}"
OUTPUT = ROOT/'AssetFoundry/out'/JOB_ID
OUTPUT.mkdir(parents=True, exist_ok=True)
print('job', JOB_ID, 'assets', ASSET_IDS)

In [ ]:
def generate_proposal(spec, output_dir):
    # Replace with the chosen GPU generator adapter.
    # Contract: write proposal.glb and provenance.json.
    raise NotImplementedError('Install and configure a generator adapter first')

def refine_topology(spec, output_dir):
    # Production stages: component pruning, manifold repair, normal repair,
    # category-aware remesh/decimation, UV unwrap, PBR bake, LODs, collision.
    # Keep hard-surface creases for cars/houses and deformation loops for zombies.
    raise NotImplementedError('Connect Blender/PyMeshLab refinement stage')

## Optional trainable vertex-topology flow refiner
A later checkpoint can predict per-vertex velocity and edge-gating logits. Train against Chamfer distance, normal consistency, Laplacian regularization, edge-length consistency, manifold penalties, self-intersection penalties, and category-specific silhouette losses. The output contract remains identical to the deterministic refiner.

In [ ]:
# Generation loop is deliberately explicit so partial jobs are resumable.
for asset_id in ASSET_IDS:
    if asset_id not in SPECS:
        raise KeyError(asset_id)
    asset_dir = OUTPUT/asset_id
    asset_dir.mkdir(parents=True, exist_ok=True)
    print('READY:', asset_id, SPECS[asset_id]['category'], asset_dir)
    # generate_proposal(SPECS[asset_id], asset_dir)
    # refine_topology(SPECS[asset_id], asset_dir)

In [ ]:
# Upload only after local validation has succeeded.
if GCS_BUCKET:
    !gcloud storage cp --recursive {OUTPUT}/* gs://{GCS_BUCKET}/jobs/{JOB_ID}/output/
    print('Import with GitHub Actions job ID:', JOB_ID)
else:
    print('Set GCS_BUCKET to publish this batch')